In [0]:
display(dbutils.fs.ls("/Volumes/databricksretailetlworkspace/csvfiles/csvdata"))

path,name,size,modificationTime
dbfs:/Volumes/databricksretailetlworkspace/csvfiles/csvdata/customer.csv,customer.csv,252,1788067663000
dbfs:/Volumes/databricksretailetlworkspace/csvfiles/csvdata/product.csv,product.csv,326,1788067663000
dbfs:/Volumes/databricksretailetlworkspace/csvfiles/csvdata/sales1.csv,sales1.csv,400,1788067663000


BRONZE LAYER


In [0]:
customer_df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/databricksretailetlworkspace/csvfiles/csvdata/customer.csv")
display(customer_df)

customer_id,customer_name,city,age
C001,Rahul,Chennai,25
C002,Priya,Hyderabad,30
C003,Arun,Bangalore,28
C004,Divya,Pune,26
C005,Karthik,Delhi,35
C006,Sneha,Mumbai,24
C007,Ravi,Chennai,29
C008,Anitha,null,27
C003,Arun,Bangalore,28
C009,null,Kolkata,32


In [0]:
product_df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/databricksretailetlworkspace/csvfiles/csvdata/product.csv")
display(product_df)

product_id,product_name,category,price
P001,Laptop,Electronics,55000
P002,Mobile,Electronics,20000
P003,Shoes,Fashion,3000
P004,Watch,Accessories,5000
P005,Headphones,Electronics,2500
P006,Tshirt,Fashion,1200
P007,Tablet,Electronics,30000
P008,Bag,Accessories,2000
P005,Headphones,Electronics,2500
P009,null,Fashion,1500


In [0]:
sales_df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/databricksretailetlworkspace/csvfiles/csvdata/sales1.csv")
display(sales_df)

order_id,customer_id,product_id,quantity,order_date
1,C001,P001,1,2026-01-01
2,C002,P002,2,2026-01-02
3,C003,P003,3,2026-01-03
4,C004,P004,1,2026-01-04
5,C005,P005,4,2026-01-05
6,C006,P006,2,2026-01-06
7,C007,P007,1,2026-01-07
8,C008,P008,2,2026-01-08
9,C001,P002,0,2026-01-09
10,C009,P003,-2,2026-01-10


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
customer_df.write.format("delta").mode("overwrite").saveAsTable("bronze.customer_raw")

In [0]:
product_df.write.format("delta").mode("overwrite").saveAsTable("bronze.product_raw")

In [0]:
sales_df.write.format("delta").mode("overwrite").saveAsTable("bronze.sales_raw")

SILVER LAYER

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
customer_bronze=spark.table("bronze.customer_raw")
display(customer_bronze)


customer_id,customer_name,city,age
C001,Rahul,Chennai,25
C002,Priya,Hyderabad,30
C003,Arun,Bangalore,28
C004,Divya,Pune,26
C005,Karthik,Delhi,35
C006,Sneha,Mumbai,24
C007,Ravi,Chennai,29
C008,Anitha,null,27
C003,Arun,Bangalore,28
C009,null,Kolkata,32


In [0]:
product_bronze=spark.table("bronze.product_raw")
display(product_bronze)


product_id,product_name,category,price
P001,Laptop,Electronics,55000
P002,Mobile,Electronics,20000
P003,Shoes,Fashion,3000
P004,Watch,Accessories,5000
P005,Headphones,Electronics,2500
P006,Tshirt,Fashion,1200
P007,Tablet,Electronics,30000
P008,Bag,Accessories,2000
P005,Headphones,Electronics,2500
P009,null,Fashion,1500


In [0]:
sales_bronze=spark.table("bronze.sales_raw")
display("sales_bronze")
sales_bronze.count()

'sales_bronze'

12

In [0]:
sales_bronze.show()

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+----------+--------+----------+
|       1|       C001|      P001|       1|2026-01-01|
|       2|       C002|      P002|       2|2026-01-02|
|       3|       C003|      P003|       3|2026-01-03|
|       4|       C004|      P004|       1|2026-01-04|
|       5|       C005|      P005|       4|2026-01-05|
|       6|       C006|      P006|       2|2026-01-06|
|       7|       C007|      P007|       1|2026-01-07|
|       8|       C008|      P008|       2|2026-01-08|
|       9|       C001|      P002|       0|2026-01-09|
|      10|       C009|      P003|      -2|2026-01-10|
|       5|       C005|      P005|       4|2026-01-05|
|      11|       C020|      P001|       1|2026-01-11|
+--------+-----------+----------+--------+----------+



clean customer data

In [0]:
from pyspark.sql.functions import *
customer_silver=customer_bronze.dropDuplicates()
customer_silver=customer_silver.fillna({"customer_name":"Unknown","city":"unknown"})
customer_silver.show()

+-----------+-------------+---------+---+
|customer_id|customer_name|     city|age|
+-----------+-------------+---------+---+
|       C004|        Divya|     Pune| 26|
|       C008|       Anitha|  unknown| 27|
|       C001|        Rahul|  Chennai| 25|
|       C003|         Arun|Bangalore| 28|
|       C007|         Ravi|  Chennai| 29|
|       C005|      Karthik|    Delhi| 35|
|       C006|        Sneha|   Mumbai| 24|
|       C002|        Priya|Hyderabad| 30|
|       C009|      Unknown|  Kolkata| 32|
+-----------+-------------+---------+---+



In [0]:
customer_silver.count()

9

cleaning product data


In [0]:
product_silver=product_bronze.dropDuplicates()
product_silver=product_silver.fillna({"product_name":"unknown"})
product_silver.count()

9

cleaning sales data

In [0]:
sales_silver=sales_bronze.dropDuplicates()
sales_silver=sales_silver.filter(col("quantity")>0)
sales_silver.count()

9

In [0]:
customer_silver.write.format("delta").mode("overwrite").saveAsTable("silver.customer_clean")
product_silver.write.format("delta").mode("overwrite").saveAsTable("silver.product_clean")
sales_silver.write.format("delta").mode("overwrite").saveAsTable("silver.sales_clean")

GOLD LAYER

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
customer_clean=spark.table("silver.customer_clean")
product_clean=spark.table("silver.product_clean")
sales_clean=spark.table("silver.sales_clean")

joining sales with customer

In [0]:
sales_customer=sales_clean.join(customer_clean,"customer_id","inner")

adding product details

In [0]:
retail_gold=sales_customer.join(product_clean,"product_id","inner")

creating revenue column

In [0]:
from pyspark.sql.functions import *
retail_gold=retail_gold.withColumn("revenue",col("price")*col("quantity"))

In [0]:
retail_gold.count()

8

In [0]:
retail_gold.write.format("delta").mode("overwrite").saveAsTable("gold.retail_sales")

VISUALIZATIONS

total revenue

In [0]:
from pyspark.sql.functions import sum
total_revenue=retail_gold.select(sum("revenue").alias("Total Revenue"))
display(total_revenue)

Total Revenue
155400


total order count

In [0]:
from pyspark.sql.functions import count
total_count=retail_gold.select(count("order_id").alias("Total Orders"))
display(total_count)

Total Orders
8


top selling products

In [0]:
top_products=retail_gold.groupBy("product_name").sum("quantity").orderBy("sum(quantity)",ascending=False)
display(top_products)

product_name,sum(quantity)
Headphones,4
Shoes,3
Mobile,2
Tshirt,2
Bag,2
Laptop,1
Tablet,1
Watch,1


Databricks visualization. Run in Databricks to view.

revenue by category

In [0]:
category_revenue=retail_gold.groupBy("category").sum("revenue").orderBy("sum(revenue)",ascending=False)
display(category_revenue)

category,sum(revenue)
Electronics,135000
Fashion,11400
Accessories,9000


Databricks visualization. Run in Databricks to view.